# Lecture 12

Today's Topics:

- Smoothing: why and how
- Simple average and mean squared error (MSE)
- Moving averages in pandas and in Altair (`transform_window`)
- Weighted moving average
- Forecasting with moving averages and the lag problem
- Exponential smoothing: single, double (Holt), and triple (Holt-Winters)
- Forecasting real stock prices

In [1]:
import warnings
from statsmodels.tools.sm_exceptions import ConvergenceWarning
warnings.filterwarnings("ignore", category=ConvergenceWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
import pandas as pd
import altair as alt
import yfinance as yf
from statsmodels.tsa.api import SimpleExpSmoothing, Holt, ExponentialSmoothing

alt.data_transformers.disable_max_rows()
np.random.seed(42)  # reproducibility

We'll reuse a few helpers from last lecture, plus a new one, `overlay`, that draws several series on the same axes with a legend. The first series is always drawn in gray as the raw data, and the rest get colors.

In [2]:
def get_close(ticker, start, end):
    """Download daily closing prices and return a flat DataFrame with Date and Close columns."""
    data = yf.download(ticker, start=start, end=end, auto_adjust=True, progress=False)
    close = data["Close"]
    if isinstance(close, pd.DataFrame):   # newer yfinance: one column per ticker
        close = close[ticker]
    return close.rename("Close").reset_index()


def run_sequence_plot(x, y, title, xlabel="Time", ylabel="Values"):
    df = pd.DataFrame({"x": x, "y": y})
    return alt.Chart(df).mark_line(color="black", strokeWidth=1.2).encode(
        x=alt.X("x:Q", title=xlabel),
        y=alt.Y("y:Q", title=ylabel, scale=alt.Scale(zero=False)),
        tooltip=[alt.Tooltip("x:Q", title=xlabel), alt.Tooltip("y:Q", format=".2f", title=ylabel)],
    ).properties(width=700, height=200, title=title)


SERIES_COLORS = ["#9e9e9e", "#d62728", "#1f77b4", "#2ca02c", "#ff7f0e", "#9467bd"]

def overlay(x, series, title, xlabel="Time", ylabel="Values"):
    """Plot several series (a dict of name -> values) on shared axes. The first one is drawn in gray."""
    names = list(series)
    df = pd.DataFrame({"x": x, **series}).melt("x", var_name="series", value_name="value")
    return alt.Chart(df).mark_line(strokeWidth=2).encode(
        x=alt.X("x:Q", title=xlabel),
        y=alt.Y("value:Q", title=ylabel, scale=alt.Scale(zero=False)),
        color=alt.Color("series:N", title=None,
                        scale=alt.Scale(domain=names, range=SERIES_COLORS[:len(names)])),
        tooltip=[alt.Tooltip("x:Q", title=xlabel), "series:N", alt.Tooltip("value:Q", format=".2f")],
    ).properties(width=700, height=250, title=title)

## Smoothing

- A data collection process is often affected by noise.
- If the noise is strong, it can hide useful patterns in the data.
- **Smoothing** recovers those patterns by *filtering out the noise*.
- It can also be used to *make forecasts* by projecting the recovered patterns into the future.

We start by generating some stationary data (white noise, from last lecture).

In [3]:
time = np.arange(100)
noise = np.random.normal(loc=0, scale=1.0, size=len(time))

run_sequence_plot(time, noise, "Stationary time series")

alt.Chart(...)

## Simple smoothing techniques

The simplest smoothing techniques are:

1. Simple average
2. Moving average
3. Weighted moving average

### Simple average

- The most conservative way to represent the stationary data above is through its **mean**.
- The mean can also be used to predict future values of the series.
- This representation is called the *simple average*.

In [4]:
noise_avg = np.full(len(time), noise.mean())   # the mean, repeated for every time step

overlay(time, {"Data": noise, "Mean": noise_avg}, "Simple average of stationary data")

alt.Chart(...)

### Mean squared error (MSE)

The mean looks like a reasonable summary here, but we need a way to measure how far off an estimate is. A common choice is the **Mean Squared Error**:

$$MSE = \frac{1}{T}\sum_{t=1}^{T} \left(X(t) - \hat{X}(t)\right)^2$$

where $X(t)$ and $\hat{X}(t)$ are the true and estimated values at time $t$.

![MSE Visualization](https://raw.githubusercontent.com/sacredfuel/workshops/main/tsa-notebooks/media/MSEVisualization.gif)

Our `mse` function flattens both inputs with `.ravel()` first. Without that, a column of shape `(n, 1)` minus a row of shape `(n,)` would silently broadcast into an `n × n` matrix and give a wrong answer with no error. (This easily happens with data from `yfinance`.)

In [5]:
def mse(observations, estimates):
    observations = np.asarray(observations, dtype=float).ravel()
    estimates = np.asarray(estimates, dtype=float).ravel()
    assert len(observations) == len(estimates), "Arrays must be of equal length"
    return np.mean((observations - estimates) ** 2)

y_true = np.array([2, 3, 1, 4, 3, 5, 4, 6, 5, 7])
y_pred = np.array([2.5, 1.0, 3.5, 3.8, 6.0, 4.5, 5.5, 4.0, 6.5, 7.5])
print(f"MSE: {mse(y_true, y_pred):.3f}")

MSE: 2.854


Next, we add a **trend** to our stationary series and try the simple average again.

In [6]:
trend = 2.0 * time + noise
trend_avg = np.full(len(time), trend.mean())

print(f"Simple average MSE: {mse(trend, trend_avg):.2f}")
overlay(time, {"Data": trend, "Mean": trend_avg}, "Simple average of data with trend")

Simple average MSE: 3338.46


alt.Chart(...)

A single flat line can't follow a trend. We need something that adapts to *local* changes in the data: the **moving average**.

### Moving average (MA)

- First, pick a window size. We'll use 3.
- Compute the average of the first three values.
- Slide the window by one step and average the next three values.
- Repeat until the end of the series.

![MA](https://raw.githubusercontent.com/sacredfuel/workshops/main/tsa-notebooks/media/MA3.gif)

In pandas, this is `.rolling(window).mean()`. There are two ways to line up the result with time:

- **Trailing** (default): the value at time $t$ is the average of $t-2, t-1, t$. It only uses the past, so it's what we use for **forecasting**, but it lags behind the data.
- **Centered** (`center=True`): the value at time $t$ is the average of $t-1, t, t+1$. It uses the future too, so it's for **describing** the data, not forecasting.

In [7]:
trend_series = pd.Series(trend)

ma_trailing = trend_series.rolling(window=3).mean()
ma_centered = trend_series.rolling(window=3, center=True).mean()

pd.DataFrame({"data": trend, "trailing MA": ma_trailing, "centered MA": ma_centered}).head()

,data,trailing MA,centered MA
0,0.496714,NaN,NaN
1,1.861736,NaN,2.335379
2,4.647689,2.335379,4.677485
3,7.523030,4.677485,6.645522
4,7.765847,6.645522,8.351580


The first values are `NaN` because the window isn't full yet. To compare with the simple average, we compute the MSE only where the MA exists.

In [8]:
valid = ma_centered.notna()
print(f"Simple average MSE: {mse(trend, trend_avg):.2f}")
print(f"Moving average MSE: {mse(trend[valid], ma_centered[valid]):.2f}")

overlay(time, {"Data": trend, "Centered MA (window 3)": ma_centered}, "Moving average of data with trend")

Simple average MSE: 3338.46
Moving average MSE: 0.56


alt.Chart(...)

### Moving averages inside Altair

Altair can compute a moving average itself with `transform_window`. The `frame` argument says which rows around the current one go into the window:

- `frame=[-2, 0]`: the current row and the 2 before it (trailing, window 3).
- `frame=[-1, 1]`: one row before, the current row, and one after (centered, window 3).
- `frame=[None, 0]`: every row up to the current one (a running/cumulative average).

Unlike pandas, Altair doesn't produce `NaN`s at the edges; it averages whatever rows are available.

In [9]:
trend_df = pd.DataFrame({"t": time, "value": trend})
base = alt.Chart(trend_df).encode(x=alt.X("t:Q", title="Time"))

raw = base.mark_line(color="#9e9e9e").encode(y=alt.Y("value:Q", title="Values"))

centered_ma = base.transform_window(
    ma="mean(value)", frame=[-1, 1]
).mark_line(color="#d62728", strokeWidth=2).encode(
    y="ma:Q",
    tooltip=["t:Q", alt.Tooltip("ma:Q", format=".2f", title="moving average")],
)

(raw + centered_ma).properties(width=700, height=250, title="Centered moving average computed by Altair")

alt.LayerChart(...)

Let's add **seasonality**. This series repeats every 12 steps.

In [10]:
seasonal = 10 + 10 * np.sin(2 * np.pi * time / 12) + noise
ma_seasonal = pd.Series(seasonal).rolling(window=3, center=True).mean()

overlay(time, {"Data": seasonal, "Centered MA (window 3)": ma_seasonal}, "Moving average of seasonal data")

alt.Chart(...)

It's not perfect (the peaks and troughs get flattened a bit) but it clearly picks up the periodic pattern.

Finally, trend, seasonality, and noise together. Note we add the noise **once**: each component is built separately and then summed.

In [11]:
trend_seasonal = 2.0 * time + 10 * np.sin(2 * np.pi * time / 12) + noise
ma_trend_seasonal = pd.Series(trend_seasonal).rolling(window=3, center=True).mean()

overlay(time, {"Data": trend_seasonal, "Centered MA (window 3)": ma_trend_seasonal},
        "Moving average of data with trend, seasonality, and noise")

alt.Chart(...)

The moving average picks up the key patterns in these toy datasets, but it has several limitations:

1. **Equal weights.** MA gives the same importance to every value in the window, regardless of how recent it is, so it's slow to react to recent changes.
2. **The window size is a choice.** A small window lets noise through; a large window oversmooths and misses short-term changes.
3. **It doesn't model trend or seasonality.** It only averages them, which leads to poor forecasts when those components matter.

### Weighted moving average (WMA)

- A WMA gives recent observations **more weight** than older ones.
- This makes intuitive sense: in the stock market, today's price is a good predictor of tomorrow's price.
- There are many ways to choose the weights ([see here](https://www.numerical.recipes/whp/notes/movingaverages.pdf)). In this example we use 0.1, 0.3, and 0.6 (oldest to newest). They add up to 1, so the result is still an average.

![WMA3](https://raw.githubusercontent.com/sacredfuel/workshops/main/tsa-notebooks/media/WMA3.gif)

`rolling(...).apply(...)` lets us run any function on each window. With `raw=True`, each window arrives as a NumPy array, oldest value first.

In [12]:
weights = np.array([0.1, 0.3, 0.6])   # oldest -> newest, sums to 1

ts = pd.Series(trend_seasonal)
ma3 = ts.rolling(window=3).mean()
wma3 = ts.rolling(window=3).apply(lambda window: np.dot(window, weights), raw=True)

To compare them fairly, we use each one as a **one-step-ahead forecast**: the smoothed value at time $t$ is our prediction for time $t+1$. `.shift(1)` moves each value forward one step so it lines up with the value it's predicting.

In [13]:
ma_forecast = ma3.shift(1)
wma_forecast = wma3.shift(1)

valid = wma_forecast.notna()
print(f"MA  one-step forecast MSE: {mse(ts[valid], ma_forecast[valid]):.2f}")
print(f"WMA one-step forecast MSE: {mse(ts[valid], wma_forecast[valid]):.2f}")

window = slice(30, 61)   # zoom in on a few cycles
overlay(time[window], {"Data": ts[window], "MA forecast": ma_forecast[window], "WMA forecast": wma_forecast[window]},
        "One-step forecasts: MA vs WMA (zoomed in)")

MA  one-step forecast MSE: 62.36
WMA one-step forecast MSE: 36.90


alt.Chart(...)

Because it leans on the most recent value, the WMA follows the data more closely, and its forecast error is lower.

### Forecasting with MA

The forecast for the next time step is the average of the last $P$ observations:

$$\hat{X}(t+1) = \frac{X(t) + X(t-1) + \dots + X(t-P+1)}{P}$$

where $P$ is the window size.

Let's take the series $X = [1, 2, 4, 8, 16, 32, 64]$ and forecast each step (including one step past the end) with $P = 3$.

In [14]:
x = np.array([1, 2, 4, 8, 16, 32, 64], dtype=float)
P = 3

actual = np.append(x, np.nan)                                 # one extra step to forecast into
ma_x_forecast = pd.Series(actual).rolling(P).mean().shift(1)  # mean of the previous P values

forecast_df = pd.DataFrame({"step": np.arange(len(actual)), "Actual": actual, "MA forecast": ma_x_forecast})
forecast_df

,step,Actual,MA forecast
0,0,1.0,NaN
1,1,2.0,NaN
2,2,4.0,NaN
3,3,8.0,2.333333
4,4,16.0,4.666667
5,5,32.0,9.333333
6,6,64.0,18.666667
7,7,NaN,37.333333


In [15]:
alt.Chart(forecast_df).transform_fold(
    ["Actual", "MA forecast"], as_=["series", "value"]
).mark_line(point=True, strokeWidth=2).encode(
    x=alt.X("step:Q", title="Time step"),
    y=alt.Y("value:Q", title="Value"),
    color=alt.Color("series:N", title=None, scale=alt.Scale(range=["#333333", "#d62728"])),
    strokeDash=alt.StrokeDash("series:N", scale=alt.Scale(range=[[1, 0], [6, 4]])),
    tooltip=["step:Q", "series:N", alt.Tooltip("value:Q", format=".2f")],
).properties(width=700, height=250, title="MA forecast of exponential growth (P = 3)")

alt.Chart(...)

- The MA forecast **lags behind** the actual signal.
- It can't keep up with changes in the trend, and the lag carries over into its forecasts.

How large is the lag? After how many steps does a change in the signal show up in the forecast?

The answer is $\frac{P+1}{2}$:

- The average of the last $P$ values is centered $\frac{P-1}{2}$ steps behind the latest observation.
- The forecast is for one step *ahead*, which adds one more step: $\frac{P-1}{2} + 1 = \frac{P+1}{2}$.
- For example, averaging the last 5 values gives a lag of $\frac{5+1}{2} = 3$ steps.

The lag grows with the window size.

### Responsiveness vs. robustness to noise

- A smaller window gives a more responsive model with less lag.
- But a window that's too small chases the noise instead of the pattern.
- There's a tradeoff between *responsiveness* and *robustness to noise*, and the right window depends on the data and the problem.

Let's see this on data with trend, seasonality, and a lot more noise.

In [16]:
noisy = 10 + 2.75 * time + 20 * np.sin(time * 0.25) + np.random.normal(loc=0, scale=8.0, size=len(time))

run_sequence_plot(time, noisy, "Noisy data")

alt.Chart(...)

First, a static comparison of four window sizes with **small multiples**: the same chart repeated in a grid, one panel per window size. We build a long DataFrame with one block of rows per window, then use `.facet(..., columns=2)`.

In [17]:
blocks = []
for w in [3, 5, 9, 19]:
    blocks.append(pd.DataFrame({
        "t": time,
        "value": noisy,
        "ma": pd.Series(noisy).rolling(w, center=True).mean(),
        "label": f"Window {w}",
    }))
multi = pd.concat(blocks)

raw_line = alt.Chart().mark_line(color="#9e9e9e").encode(
    x=alt.X("t:Q", title="Time"), y=alt.Y("value:Q", title="Values")
)
ma_line = alt.Chart().mark_line(color="#d62728", strokeWidth=2.5).encode(x="t:Q", y="ma:Q")

alt.layer(raw_line, ma_line, data=multi).properties(width=330, height=180).facet(
    facet=alt.Facet("label:N", title=None, sort=list(multi["label"].unique())), columns=2
)

alt.FacetChart(...)

The larger the window, the smoother the line. That removes the noise, but eventually it smooths away the real signal as well.

Small multiples are good for print, but in a notebook we can do better with an **interactive slider**.

- `alt.param(...)` creates a variable that the chart can read.
- `bind=alt.binding_range(...)` attaches it to a slider widget below the chart.
- `transform_filter(alt.datum.window == window_size)` keeps only the rows for the selected window.

We precompute the moving average for every odd window from 1 to 31, and the slider picks which one to show.

In [18]:
windows = range(1, 32, 2)
smoothed = pd.concat([
    pd.DataFrame({"t": time, "window": w, "ma": pd.Series(noisy).rolling(w, center=True).mean()})
    for w in windows
])

window_size = alt.param(
    name="window_size", value=5,
    bind=alt.binding_range(min=1, max=31, step=2, name="Window size "),
)

raw_noisy = alt.Chart(pd.DataFrame({"t": time, "value": noisy})).mark_line(color="#9e9e9e").encode(
    x=alt.X("t:Q", title="Time"), y=alt.Y("value:Q", title="Values")
)
ma_selected = alt.Chart(smoothed).transform_filter(
    alt.datum.window == window_size
).mark_line(color="#d62728", strokeWidth=2.5).encode(
    x="t:Q", y="ma:Q", tooltip=["t:Q", alt.Tooltip("ma:Q", format=".2f")]
)

(raw_noisy + ma_selected).add_params(window_size).properties(
    width=700, height=280, title="Centered moving average: drag the slider to change the window"
)

alt.LayerChart(...)

---

## Exponential smoothing

There are three key exponential smoothing techniques:

| Type | Captures trend | Captures seasonality |
|:----:|:--------------:|:--------------------:|
| Single exponential smoothing | ❌ | ❌ |
| Double exponential smoothing (Holt) | ✅ | ❌ |
| Triple exponential smoothing (Holt-Winters) | ✅ | ✅ |

The basic formulation is:

$$S(t) = \alpha X(t) + (1-\alpha)S(t-1)$$

where:

- $S(t)$ is the smoothed value at time $t$,
- $\alpha$ is the smoothing constant, between 0 and 1,
- $X(t)$ is the value of the series at time $t$.

![EMA](https://raw.githubusercontent.com/sacredfuel/workshops/main/tsa-notebooks/media/EMA.gif)

It's a weighted moving average in disguise. If you expand the recursion, the weight on a value $k$ steps in the past is $\alpha(1-\alpha)^k$:

- Recent values get the most weight, and the weights shrink exponentially (hence the name).
- *Every* past value contributes a little, so there's no window size to choose, only $\alpha$.
- A large $\alpha$ reacts quickly to new data; a small $\alpha$ is smoother but lags more.

The formula is short enough to implement directly.

In [19]:
def exponential_smoothing(x, alpha):
    smoothed = np.empty(len(x))
    smoothed[0] = x[0]                                   # start from the first observation
    for t in range(1, len(x)):
        smoothed[t] = alpha * x[t] + (1 - alpha) * smoothed[t - 1]
    return smoothed

pandas has this built in as `.ewm(alpha=...).mean()` (exponentially weighted mean). With `adjust=False` it uses exactly the recursion above, so the two should match.

In [20]:
mine = exponential_smoothing(noisy, alpha=0.3)
pandas_version = pd.Series(noisy).ewm(alpha=0.3, adjust=False).mean()

print("Same result:", np.allclose(mine, pandas_version))

Same result: True


Same idea as before: precompute the smoothed series for a range of $\alpha$ values and use a slider to choose between them.

The slider values are decimals like 0.15, which can't always be represented exactly as floating-point numbers. So instead of testing `alpha == slider`, the filter checks that they're within 0.001 of each other.

In [21]:
alphas = np.round(np.arange(0.05, 1.0, 0.05), 2)
ses_curves = pd.concat([
    pd.DataFrame({"t": time, "alpha": a, "smoothed": pd.Series(noisy).ewm(alpha=a, adjust=False).mean()})
    for a in alphas
])

alpha_slider = alt.param(
    name="alpha", value=0.3,
    bind=alt.binding_range(min=0.05, max=0.95, step=0.05, name="alpha "),
)

ses_selected = alt.Chart(ses_curves).transform_filter(
    alt.expr.abs(alt.datum.alpha - alpha_slider) < 0.001
).mark_line(color="#d62728", strokeWidth=2.5).encode(
    x="t:Q", y="smoothed:Q", tooltip=["t:Q", alt.Tooltip("smoothed:Q", format=".2f")]
)

(raw_noisy + ses_selected).add_params(alpha_slider).properties(
    width=700, height=280, title="Exponential smoothing: drag the slider to change alpha"
)

alt.LayerChart(...)

**Double** exponential smoothing (Holt's method) adds a second equation that tracks the **trend**, so its forecasts can keep going up or down.

**Triple** exponential smoothing (Holt-Winters) adds a third equation that tracks the **seasonal** pattern.

We'll let `statsmodels` do the math and focus on comparing the forecasts.

### Forecasting setup

- We hold out the last 12 values of `trend_seasonal` as the **test set**, one full seasonal cycle.
- Each model is fit on the training set only, then asked to forecast the next 12 steps.
- We compare the forecasts to the test set with MSE.

In [22]:
horizon = 12
train, test = trend_seasonal[:-horizon], trend_seasonal[-horizon:]
t_train, t_test = time[:-horizon], time[-horizon:]

forecasts = {}   # model name -> its 12 predictions

A helper to plot the training data, the actual test values, and one or more forecasts. Forecasts are drawn dashed so they're easy to tell apart from real data.

In [23]:
def forecast_chart(x_train, y_train, x_test, y_test, predictions, title, x_type="Q", history=None):
    """Plot train data, actual test data, and a dict of forecasts (name -> values)."""
    if history is not None:                  # optionally show only the last `history` training points
        x_train, y_train = x_train[-history:], y_train[-history:]
    frames = [
        pd.DataFrame({"x": x_train, "value": np.ravel(y_train), "series": "Train"}),
        pd.DataFrame({"x": x_test, "value": np.ravel(y_test), "series": "Actual"}),
    ]
    for name, values in predictions.items():
        frames.append(pd.DataFrame({"x": x_test, "value": np.ravel(values), "series": name}))
    df = pd.concat(frames)

    names = ["Train", "Actual"] + list(predictions)
    colors = ["#9e9e9e", "#333333"] + ["#d62728", "#1f77b4", "#2ca02c", "#ff7f0e", "#9467bd"][:len(predictions)]
    dashes = [[1, 0], [1, 0]] + [[6, 4]] * len(predictions)

    return alt.Chart(df).mark_line(strokeWidth=2).encode(
        x=alt.X(f"x:{x_type}", title="Time" if x_type == "Q" else "Date"),
        y=alt.Y("value:Q", title="Values", scale=alt.Scale(zero=False)),
        color=alt.Color("series:N", title=None, scale=alt.Scale(domain=names, range=colors)),
        strokeDash=alt.StrokeDash("series:N", scale=alt.Scale(domain=names, range=dashes)),
        tooltip=[alt.Tooltip(f"x:{x_type}", title="Time"), "series:N", alt.Tooltip("value:Q", format=".2f")],
    ).properties(width=700, height=280, title=title)

### Simple average

A very crude model, but a useful **baseline**: any model we try should do much better than this.

In [24]:
forecasts["Simple average"] = np.full(horizon, train.mean())

print(f"MSE: {mse(test, forecasts['Simple average']):.2f}")
forecast_chart(t_train, train, t_test, test, {"Simple average": forecasts["Simple average"]},
               f"Simple average - MSE: {mse(test, forecasts['Simple average']):.2f}")

MSE: 10060.43


alt.Chart(...)

### Single exponential smoothing

`fit()` chooses $\alpha$ automatically by minimizing the error on the training data. `initialization_method="estimated"` lets it estimate the starting value too.

In [25]:
single = SimpleExpSmoothing(train, initialization_method="estimated").fit()
forecasts["Single"] = single.forecast(horizon)

print(f"Fitted alpha: {single.params['smoothing_level']:.3f}")
print(f"MSE: {mse(test, forecasts['Single']):.2f}")
forecast_chart(t_train, train, t_test, test, {"Single": forecasts["Single"]},
               f"Single exponential smoothing - MSE: {mse(test, forecasts['Single']):.2f}")

Fitted alpha: 1.000
MSE: 125.57


alt.Chart(...)

- Better than the simple average, but still crude.
- The forecast is a **flat line**: single exponential smoothing has no notion of trend or seasonality, so its best guess for every future step is the last smoothed value.
- The fitted $\alpha$ is (close to) 1. With $\alpha = 1$, $S(t) = X(t)$: the "smoothed" value is just the last observation. For a series that keeps rising, the optimizer finds that trusting the newest value is the best it can do without a trend term.

### Double exponential smoothing (Holt)

In [26]:
double = Holt(train, initialization_method="estimated").fit()
forecasts["Double (Holt)"] = double.forecast(horizon)

print(f"MSE: {mse(test, forecasts['Double (Holt)']):.2f}")
forecast_chart(t_train, train, t_test, test, {"Double (Holt)": forecasts["Double (Holt)"]},
               f"Double exponential smoothing - MSE: {mse(test, forecasts['Double (Holt)']):.2f}")

MSE: 360.70


alt.Chart(...)

- Holt's method picks up the trend: the forecast now rises with the data.
- It still misses the seasonal ups and downs, and here that costs it: its MSE is actually *worse* than single exponential smoothing.
- Why? The training data ends on a seasonal **upswing**, so the most recent slope is much steeper than the true trend. Holt treats that slope as the trend and extends it, overshooting as the season turns down.
- Lesson: a model that captures *more* components isn't automatically better. A model with a trend but no seasonality can be misled by the seasonal pattern.

### Triple exponential smoothing (Holt-Winters)

We tell it the seasonal period: 12 steps, since we built the data with `np.sin(2 * np.pi * time / 12)`. If we didn't know it, we could find it with the FFT from last lecture.

In [27]:
triple = ExponentialSmoothing(
    train, trend="add", seasonal="add", seasonal_periods=12, initialization_method="estimated"
).fit()
forecasts["Triple (Holt-Winters)"] = triple.forecast(horizon)

print(f"MSE: {mse(test, forecasts['Triple (Holt-Winters)']):.2f}")
forecast_chart(t_train, train, t_test, test, {"Triple (Holt-Winters)": forecasts["Triple (Holt-Winters)"]},
               f"Triple exponential smoothing - MSE: {mse(test, forecasts['Triple (Holt-Winters)']):.2f}")

MSE: 0.47


alt.Chart(...)

Holt-Winters captures both the trend and the seasonality, so it's clearly the right model for this data.

### Comparing the models

All four forecasts on one chart (zoomed in to the last few cycles), and their errors.

In [28]:
forecast_chart(t_train, train, t_test, test, forecasts, "All forecasts", history=36)

alt.Chart(...)

In [29]:
results = pd.DataFrame({
    "model": list(forecasts),
    "MSE": [mse(test, preds) for preds in forecasts.values()],
})
results

,model,MSE
0,Simple average,10060.433333
1,Single,125.567022
2,Double (Holt),360.695357
3,Triple (Holt-Winters),0.471196


The errors span several orders of magnitude, so we use a **log scale** on the x-axis; otherwise the best model would be squashed against zero.

We use points instead of bars because bars start at zero, and zero can't be placed on a log scale (log(0) is undefined).

In [30]:
alt.Chart(results).mark_point(filled=True, size=120).encode(
    x=alt.X("MSE:Q", scale=alt.Scale(type="log"), title="Test MSE (log scale)"),
    y=alt.Y("model:N", sort="x", title=None),
    tooltip=["model:N", alt.Tooltip("MSE:Q", format=",.2f")],
).properties(width=600, height=160, title="Forecast error by model (lower is better)")

alt.Chart(...)

---

## Real data: stock prices

Let's apply these ideas to Apple's stock price.

A common use of exponential smoothing in finance is the **exponential moving average** (EMA). Traders usually describe it by its *span* (roughly, how many days it averages over) instead of $\alpha$. The two are related by $\alpha = \frac{2}{\text{span} + 1}$, and pandas accepts either one.

In [31]:
aapl = get_close("AAPL", "2023-08-20", "2025-08-20")
aapl["EMA 20"] = aapl["Close"].ewm(span=20, adjust=False).mean()
aapl["EMA 50"] = aapl["Close"].ewm(span=50, adjust=False).mean()

alt.Chart(aapl).transform_fold(
    ["Close", "EMA 20", "EMA 50"], as_=["series", "price"]
).mark_line(strokeWidth=1.8).encode(
    x=alt.X("Date:T", title="Date"),
    y=alt.Y("price:Q", title="Price (USD)", scale=alt.Scale(zero=False)),
    color=alt.Color("series:N", title=None,
                    scale=alt.Scale(domain=["Close", "EMA 20", "EMA 50"], range=SERIES_COLORS[:3])),
    tooltip=[alt.Tooltip("Date:T"), "series:N", alt.Tooltip("price:Q", format="$.2f")],
).properties(width=700, height=300, title="AAPL closing price with 20- and 50-day EMAs").interactive(bind_y=False)

alt.Chart(...)

The 20-day EMA hugs the price closely; the 50-day EMA is smoother but lags further behind. It's the same responsiveness-vs-noise tradeoff we saw with window sizes.

### Forecasting the stock

- We hold out the last 20 trading days (about one month) as the test set.
- We compare single and double exponential smoothing against a **naive** baseline: "tomorrow's price will be the same as today's", repeated for all 20 days.
- We **don't** use Holt-Winters here. Stock prices have no reliable seasonal cycle, and inventing a seasonal period (for example by running the FFT on prices) just fits noise.

In [32]:
prices = aapl["Close"].to_numpy()
dates = aapl["Date"]

stock_horizon = 20
stock_train, stock_test = prices[:-stock_horizon], prices[-stock_horizon:]
date_train, date_test = dates[:-stock_horizon], dates[-stock_horizon:]

stock_forecasts = {
    "Naive (last value)": np.full(stock_horizon, stock_train[-1]),
    "Single": SimpleExpSmoothing(stock_train, initialization_method="estimated").fit().forecast(stock_horizon),
    "Double (Holt)": Holt(stock_train, initialization_method="estimated").fit().forecast(stock_horizon),
}

for name, preds in stock_forecasts.items():
    print(f"{name:20s} MSE: {mse(stock_test, preds):8.2f}")

Naive (last value)   MSE:   134.81
Single               MSE:   134.81
Double (Holt)        MSE:   120.60


In [33]:
forecast_chart(date_train, stock_train, date_test, stock_test, stock_forecasts,
               "AAPL: 20-day forecasts", x_type="T", history=120)

alt.Chart(...)

Don't be surprised if the naive forecast is hard to beat, or if it wins outright.

- Daily stock prices behave a lot like the **random walk** from last lecture: tomorrow's price is today's price plus an unpredictable step.
- For a random walk, the best forecast really is the last value, and the fitted single exponential smoothing usually ends up with $\alpha$ close to 1, which makes it nearly identical to the naive forecast.
- Holt's trend forecast can look better or worse depending on whether the recent trend happens to continue.

This is why every forecasting comparison should include a simple baseline.
